# EXTRA EXERCISE 3

In a chemical process it is necessary to keep constant the pH of a compound. Measurements are made every hour. Data acquired over the first 48 hours are reported in `pH.csv`.

Identify and fit a model for the data.

In [ ]:
# Import the necessary libraries
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
import qdatoolkit as qda

# Import the dataset
data = pd.read_csv('../../Data/pH.csv')

# Inspect the dataset
data.head()

> ### Solution
>
> Let's first check if:
> - The data are random.
> - (If random) The data are normally distributed.

In [ ]:
# Plot the data first
plt.plot(data['pH'], 'o-')
plt.xlabel('Index')
plt.ylabel('PH measuremet')
plt.title('Time series plot of pH measurements')
plt.grid()
plt.show()

In [ ]:
# let's check randomness of the data
qda.Assumptions(data['pH']).independence()

> The runs test gives a null p-value, this means that the data are <span style="color:red"> not random </span>. 
>
> Plot also the autocorrelation and partial autocorrelation functions of the data. Use the `plot_acf` and `plot_pacf` functions from the `statsmodels` package.

> There is a strong positive correlation. Decay of autocorrelation coefficients is not exponential.
> Based on ACF analysis, we can state that the process is non-stationary
>
> We can observe with a scatterplot the correlation between $X(t)$ and $X(t-1)$.

In [ ]:
#calculate the lag1 from data
data['lag1'] = data['pH'].shift(1)

#create scatterplot with regression line using seaborn and set axis labels
sns.regplot(x=data['lag1'], y=data['pH'], ci=None, line_kws={'color':'red', 'ls':'--'})
plt.title('Scatter plot of X(t-1) vs X(t)')
plt.xlabel('X(t-1)')
plt.ylabel('X(t)')
plt.title('Scatter plot of X(t-1) vs X(t)')
plt.grid()



> Let's apply the time difference operator to our time serie data

In [ ]:
#calculate the difference between the data and the lag1
data['diff1'] = data['pH'] - data['lag1']

plt.plot(data['diff1'], 'o-')
plt.xlabel('Index')
plt.ylabel('DIFF 1')
plt.title('Time series plot of DIFF 1')
plt.grid()
plt.show()

> Let's check if differences at lag 1 are NID

In [ ]:
#Let's calculate the p-value (exclude the first value because it is null)
qda.Assumptions(data['diff1'][1:]).independence()

In [ ]:
# Let's check lag 1 with Bartlett's test
qda.Assumptions(data['diff1'][1:]).independence(ac_test='bartlett', lag=1)

This is a typical borderline situation. At 95% confidence, we should reject the randomness assumption. In this case, we may proceed by fitting an `AR(1)` model on the transformed data by applying the difference operator, and check if residuals are NID. If residuals are NID, the resulting model would be an `ARIMA(1,1,0)`.
Another option is to accept the transformed data are barely random. If we follow this second path, we shall verify if they are also normal. Let's follow this second route (we'll see examples of `ARIMA(1,1,0)` models later on). 

In [ ]:
# Perform the Shapiro-Wilk test
qda.Assumptions(data['diff1'][1:]).normality()

> The process is modeled as a <t1 style="color:red"> RANDOM WALK </t1>
>
> **Random Walk**:
> - $Y_{t} = Y_{t-1} + \epsilon_{t} $

It is the best model? Is this the best one?

We can also try an AR(1) model directly on the original data (no application of the difference operator)


In [ ]:
x = data['lag1'][1:]
x = sm.add_constant(data['lag1'][1:]) # this command is used to consider a constant to the model, is equivalent to create and add a column of ones
y = data['pH'][1:]
model = sm.OLS(y, x).fit()

qda.summary(model)


> Note that the p-value of the constant term is 0.332. 
>
><t1 style="color:red"> Let's re-calculate the model removing the constant term </t1>

In [ ]:
x = data['lag1'][1:]
y = data['pH'][1:]
model = sm.OLS(y, x).fit()
qda.summary(model)

> <t1 style="color:red"> We have found again the random walk model! </t1>
>
> $$EXE1 = 0.9997 \cdot lag1$$
>
> Let's check assumptions on residuals:
> - Normality
> - Time independence

In [ ]:
qda.Assumptions(model.resid).normality()

In [ ]:
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')
stats.probplot(model.resid, dist="norm", plot=axs[0,0])
axs[0,0].set_title('Normal probability plot')
axs[0,1].scatter(model.fittedvalues, model.resid)
axs[0,1].set_title('Versus Fits')
fig.subplots_adjust(hspace=0.5)
axs[1,0].hist(model.resid)
axs[1,0].set_title('Histogram')
axs[1,1].plot(np.arange(1, len(model.resid)+1), model.resid, 'o-')

In [ ]:
# Let's check the residuals for randomness
qda.Assumptions(model.resid).independence()

Let's check autocorrelation at lag 1 with Bartlett.
At 5% significance level:

In [ ]:
#Let's check autocorrelation at lag1 with Bartlett's test
qda.Assumptions(model.resid).independence(plotit=False, ac_test='bartlett', lag=1)

Same result as before (indeed the model is the same, i.e., random walk)
The Bartlett's null hypothesis is barely rejected at 95% confidence but not rejected at 99% confidence.  

In [ ]:
plt.plot(data['pH'], 'o-', label='Original data')
plt.xlabel('Index') 
plt.ylabel('pH')
plt.plot(model.fittedvalues, 's--', color='red', label='Fitted values', alpha=0.5)
plt.legend()
plt.grid()
plt.show()